# A Gentle, Practical Guide to Classification and Regression for Speech Signals

This notebook explains how to set up, train, and evaluate classification and regression systems with speech data. Although the examples use speech features, the ideas generalize to many other domains.

---

## 1) The road map

Every supervised learning project follows the same arc. You begin by deciding what you want to predict and why that prediction is useful. Next you prepare a dataset, which includes collecting or loading audio, cleaning it, and turning each recording into features your models can work with (unless you want to work with raw data -- some deep learning models allow that). 

You then split the data so that the models are tuned on one subset and judged fairly on a separate subset that the model has NEVER seen. You train a few strong but simple baselines, evaluate them with the right metrics, and report your results with honest uncertainty. Only after this foundation is solid do you consider fancier models.

Three concepts will keep appearing: **independence** between training and testing (the model should not see the same speaker in both), **reproducibility** (results should be repeatable by someone else who follows your steps), and **unit of testing** (usually datasets have more than one recording per speaker, you should evaluate on a per speaker and not on a per waveform basis!)

---

## 2) Data preparation in plain language

> **Math note.** RMS normalization of an utterance $x[n]$ to target level $L_{\text{RMS}}$ uses $x'[n]=\alpha\,x[n]$ with $$\alpha = L_{\text{RMS}}/\sqrt{\tfrac{1}{N}\sum_{n=0}^{N-1} x[n]^2}$$ A simple energy‑based VAD marks frame $t$ as speech when $$\tfrac{1}{N}\sum_{m=0}^{N-1} x_t[m]^2 > \tau$$ for a threshold $\tau$; practical systems add hysteresis to avoid flicker.

Start with audio in a consistent format, ideally mono WAV at 16 kHz or higher. Keep a raw copy untouched—preprocessing is always reversible if you saved the original. Trim long silences and use a voice‑activity detector to isolate speech. Apply gentle normalization (for example, RMS normalization per utterance) so that input volumes are comparable. If you must reduce noise, do it lightly; the goal is not an aesthetically pleasing sound but a signal that preserves the cues the model will learn from. Resample everything to the same sampling rate with a high‑quality resampler.

Metadata matters. For each recording, keep a speaker ID, the device or microphone used, the date, and any prompts or tasks. These small details later let you design proper splits, check generalization, and understand failures. If multiple annotators provide labels or ratings, measure their agreement: Cohen’s $\kappa$ for categorical labels and ICC for continuous ratings are standard choices. Low agreement is a warning sign that no model will do well without clearer instructions or better data.

Augmentation is a way to teach the model to ignore nuisance variability. Adding noise at different SNRs, convolving with room impulse responses, slightly speeding up or slowing down, or applying small gain changes are all useful. The important rule is to generate augmentations **within** the training split only; you should never let an augmented version of a file leak into validation or test.

---

## 3) Features that get you moving

> **Math note.** When you pool frame‑level features $\mathbf{f}_t\in\mathbb{R}^d$ to an utterance vector, use the mean and (unbiased) standard deviation: $$\boldsymbol\mu=\tfrac{1}{N}\sum_{n=1}^N \mathbf{f}_n$$, $$\;\boldsymbol\sigma=\sqrt{\tfrac{1}{N-1}\sum_{n=1}^N (\mathbf{f}_n-\boldsymbol\mu)^{2}}$$ Concatenating $[\boldsymbol\mu;\boldsymbol\sigma]$ gives a $2d$‑dimensional representation well suited to tabular models.

You can build effective systems with simple features. For time–frequency representations, log‑Mel spectrograms and MFCCs are reliable starting points. A common recipe uses 25 ms windows with a 10 ms hop and extracts 20–40 MFCCs plus their deltas. From these, compute summary statistics per utterance such as the mean and standard deviation. Prosodic and spectral‑shape descriptors—fundamental‑frequency statistics, energy statistics, spectral centroid, bandwidth, roll‑off, flux, or a simple high‑to‑low‑frequency energy ratio—add complementary information. Later, when you want stronger performance, you can replace or augment these with pre‑trained embeddings (e.g., wav2vec2 or ECAPA) averaged over time. The modeling steps that follow remain the same.

---

## 4) Splits and cross‑validation without leakage

> **Math note.** If speaker $i$ has utterance‑level scores $\{\hat{s}_{i,u}\}_{u=1}^{U_i}$, evaluate per speaker using $\bar{s}_i=\tfrac{1}{U_i}\sum_{u=1}^{U_i} \hat{s}_{i,u}$ or some other aggregation approach.
>
> Grouped $K$‑fold CV with groups $g(n)$ (speaker IDs) constructs folds $(\mathcal{T}_k,\mathcal{V}_k)$ so that $g(\mathcal{T}_k)\cap g(\mathcal{V}_k)=\varnothing$ for all $k$, ensuring independence by construction.

The hardest mistakes to recover from are data‑splitting errors. If you cut a recording into multiple segments, all segments from the same speaker—and often from the same session or device—must live in a single split. Otherwise, the model memorizes speaker traits and appears unrealistically accurate.

A simple and dependable approach is to reserve a hold‑out test set comprising 15–20% of the speakers from the beginning and never touch it during model selection. For day‑to‑day development, use **GroupKFold** cross‑validation, where the groups are the speaker IDs. This ensures that, on each fold, the validation speakers are disjoint from the training speakers. When the cohort is small and per‑speaker generalization is critical, **Leave‑One‑Speaker‑Out** cross‑validation is a good choice. If you must tune many hyperparameters, use nested cross‑validation: the outer loop estimates generalization, and the inner loop searches hyperparameters—both grouping by speakers.

If devices or time periods differ, account for them explicitly. You might, for example, use groups that combine speaker and device, or use blocked folds by time so that you evaluate on later data than you trained on. The guiding principle is to mimic the future conditions under which the model will be used.

---

## 5) Modeling: strong baselines first

> **Math note.** Standardization inside a pipeline applies $z_j=(x_j-\mu_j)/\sigma_j$ using $(\mu_j,\sigma_j)$ estimated **only** on the training fold. 
> 
> Logistic regression minimizes $$\sum_i \log\!\big(1+e^{-y_i(\mathbf w^\top \mathbf x_i+b)}\big)+\lambda\,\Omega(\mathbf w)$$
> 
> SVM classification (hinge loss) minimizes $$\sum_i \max\{0,1-y_i(\mathbf w^\top\mathbf x_i+b)\}+\tfrac{\lambda}{2}\|\mathbf w\|_2^2$$
> 
> Boosted trees fit additive functions $$F_M(\mathbf x)=\sum_{m=1}^M \nu f_m(\mathbf x)$$ to the negative gradients of a chosen loss.

With tabular per‑utterance features, a few classic methods consistently perform well and are easy to reason about. For classification, begin with Logistic Regression, an **SVM with an RBF kernel**, a **Random Forest**, and a **Gradient‑Boosted Trees** model (such as XGBoost or LightGBM). For regression, use Linear or Ridge Regression, **SVR with an RBF kernel**, **RandomForestRegressor**, and a **Gradient‑Boosting Regressor**. These methods require little feature engineering, capture nonlinearities, and are robust to small datasets.

Always wrap preprocessing and the model in a single scikit‑learn `Pipeline`. This prevents leakage (for example, standardization parameters being estimated on the entire dataset) and simplifies cross‑validation. Tune only a handful of hyperparameters at first—`C` and `gamma` for SVMs, the number of trees and minimum leaf size for forests, and the learning rate, depth, and number of estimators for boosting. Early stopping on a validation fold helps boosted models avoid overfitting.

Aggregate predictions at the speaker level before scoring. If your unit of prediction is an utterance but decisions are made per speaker or session, average the probabilities (for classification) or the predictions (for regression) over all utterances from that speaker, and then compute metrics. This reduces variance and matches how the system will be used.

---

## 6) Evaluating classification models

> **Math note.** Consider binary classification with labels $y_i\in\{0,1\}$, predicted probabilities $p_i\in[0,1]$, and a decision threshold $t$ (e.g., $t=0.5$) giving $\hat y_i=\mathbb{1}[p_i\ge t]$.
>
> Define the confusion counts:
> $$
> \mathrm{TP}=\sum_i \mathbb{1}[y_i=1,\hat y_i=1],\quad
> \mathrm{TN}=\sum_i \mathbb{1}[y_i=0,\hat y_i=0],\quad
> \mathrm{FP}=\sum_i \mathbb{1}[y_i=0,\hat y_i=1],\quad
> \mathrm{FN}=\sum_i \mathbb{1}[y_i=1,\hat y_i=0].
> $$
>
> Then:
> $$
> \mathrm{Accuracy}=\frac{\mathrm{TP}+\mathrm{TN}}{n}
> $$
> $$
> \mathrm{Precision}=\frac{\mathrm{TP}}{\mathrm{TP}+\mathrm{FP}}
> $$
> $$
> \mathrm{Recall \: (Sensitivity)}=\frac{\mathrm{TP}}{\mathrm{TP}+\mathrm{FN}}
> $$
> $$
> \mathrm{Specificity}=\frac{\mathrm{TN}}{\mathrm{TN}+\mathrm{FP}}
> $$
>
> The $F_1$ score is the harmonic mean of precision and recall:
> $$
> \mathrm{F1}=\frac{2\,\mathrm{Precision}\cdot\mathrm{Recall}}{\mathrm{Precision}+\mathrm{Recall}}.
> $$
>
> The ROC curve uses $$\mathrm{TPR}(t)=\mathrm{Recall}(t)$$ and $$\mathrm{FPR}(t)=\frac{\mathrm{FP}(t)}{\mathrm{FP}(t)+\mathrm{TN}(t)}$$ **ROC-AUC** is the area under $(\mathrm{FPR}(t),\mathrm{TPR}(t))$.  
> The precision–recall (PR) curve uses $(\mathrm{Recall}(t),\mathrm{Precision}(t))$; **PR-AUC** is
> $$
> \mathrm{AUPRC}=\int_0^1 \mathrm{Precision}(r)\,\mathrm{d}r.
> $$
>
> For probabilistic forecasts, the **Brier score** measures calibration + sharpness:
> $$
> \mathrm{Brier}=\frac{1}{n}\sum_{i=1}^n (p_i-y_i)^2.
> $$
>
> A common threshold rule is **Youden’s $J$**:
> $$
> J(t)=\mathrm{TPR}(t)-\mathrm{FPR}(t)=\mathrm{Sensitivity}(t)+\mathrm{Specificity}(t)-1,
> $$
> choosing $t^\star=\arg\max_t J(t)$. Another is fixed sensitivity: pick the smallest $t$ such that $\mathrm{TPR}(t)\ge\tau$ (e.g., $\tau=0.90$) and report the resulting specificity.

Classification is fundamentally about *ranking* and *decision-making under a threshold*. If classes are imbalanced, accuracy can be misleading: always report threshold-free ranking metrics (**ROC-AUC**, and especially **PR-AUC** when the positive class is rare), and thresholded operating metrics (sensitivity/recall, specificity, precision, $F_1$) at a clearly stated thresholding rule (e.g., fixed sensitivity or Youden’s $J$).

When you produce probability outputs (or any score interpreted as probability), you should assess calibration. A model with strong ROC-AUC can still be poorly calibrated. The **Brier score** provides a simple scalar summary, while reliability diagrams (calibration curves) show *where* the model is over- or under-confident.

For multi-class problems, compute per-class precision/recall/$F_1$ and report **macro-$F_1$** (unweighted mean across classes) to avoid large classes dominating the score. As with regression, compute metrics at the **speaker level** (aggregate utterance probabilities per speaker before scoring), and attach bootstrap confidence intervals by resampling speakers rather than individual utterances.

---

## 7) Evaluating regression models

> **Math note.** With residuals $e_i=y_i-\hat{y}_i$: $$\mathrm{MAE}=\tfrac{1}{n}\sum_i |e_i|$$ $$\mathrm{RMSE}=\sqrt{\tfrac{1}{n}\sum_i e_i^2}$$ $$ R^2=1-\tfrac{\sum_i e_i^2}{\sum_i (y_i-\bar y)^2}$$ 
>
> For agreement, the concordance correlation coefficient is $$\mathrm{CCC}=\tfrac{2\,\sigma_{y\hat y}}{\sigma_y^2+\sigma_{\hat y}^2+(\mu_y-\mu_{\hat y})^2}$$ and Bland–Altman limits are $\bar d\pm1.96\,s_d$ with $d_i=y_i-\hat y_i$.

For continuous targets, **MAE** is a robust and interpretable primary metric, while **RMSE** penalizes larger errors and can reveal sensitivity to outliers. Report **R²** as a unitless measure of explained variance. When your target represents a human rating or a measurement with noise, complement these with an agreement measure such as the **Concordance Correlation Coefficient (CCC)** and a **Bland–Altman** plot, which shows bias and the spread of errors. As with classification, compute metrics per speaker and attach bootstrap confidence intervals.

---

## 8) Honest uncertainty with conformal prediction

Point predictions alone can overstate confidence. **Conformal prediction** adds valid uncertainty statements under minimal assumptions. For regression, split your speakers into a proper‑training subset and a calibration subset. Train the model on the former, compute absolute residuals on the latter, and take the 90th percentile if you want 90% coverage. For a new input, return the prediction plus or minus that quantile—an interval that will cover the true value about 90% of the time on new speakers drawn like the calibration ones. For classification, conformal methods output **prediction sets**: sometimes a single label, and sometimes a small set when the model is unsure. Use class‑conditional (Mondrian) variants when classes are imbalanced.

---

## 9) A narrated minimal workflow

> **Math note.** When you finally evaluate on the frozen test speakers, attach 95% confidence intervals by **speaker bootstrap**: resample speakers with replacement, recompute the metric for each bootstrap $b=1,\ldots,B$, and report the $2.5$ th and $97.5$ th percentiles of $\{M^{(b)}\}$. This communicates sampling variability without parametric assumptions.

Imagine you receive a folder of WAV files. You first decide what your target is—perhaps a binary label or a continuous score—and assemble a table with one row per utterance, columns for features, one column for the target, and one for the `speaker_id`. You freeze a list of hold‑out speakers that will constitute your test set and promise not to use them for anything until the end. You compute MFCC statistics (means and standard deviations) for each file, or you extract an embedding per utterance and average across time. You then build a scikit‑learn pipeline with a scaler and an SVM, and you evaluate it with **GroupKFold** so that speakers do not mix across folds. After the cross‑validation run, you average scores by speaker and report PR‑AUC, ROC‑AUC, and macro‑F1 with bootstrap confidence intervals. If performance is promising, you repeat the process with a Random Forest and a Gradient‑Boosted Trees model, pick whichever generalizes best, and finally evaluate once on the frozen test speakers. To communicate uncertainty for regression tasks, you add conformal intervals using a small calibration group of speakers held out from training.

Throughout this process you log random seeds, library versions, and most importantly the exact speaker splits you used. Saving a CSV with the fold assignment for every utterance makes your experiments reproducible by teammates.

---

## 10) Minimal, copy‑and‑run code with commentary

The snippets below are intentionally compact. They assume three arrays: `X` with features, `y` with labels or targets, and `speaker_id` with the grouping key for cross‑validation. Small helper functions are included for bootstrap confidence intervals and for the concordance correlation coefficient. Dummy loaders allow you to test the pipeline before wiring it to your own features.

In [ ]:
import numpy as np, pandas as pd, os, random
from sklearn.model_selection import GroupKFold

# Reproducibility
def set_seed(s=123):
    random.seed(s); np.random.seed(s)
set_seed(7)

# Bootstrap 95% CI over speakers
def ci95_by_speaker(metric_fn, y_true, y_pred, speaker_ids, B=1000):
    speakers = np.array(pd.unique(speaker_ids))
    stats = []
    for _ in range(B):
        samp = np.random.choice(speakers, size=len(speakers), replace=True)
        idx = np.isin(speaker_ids, samp)
        stats.append(metric_fn(y_true[idx], y_pred[idx]))
    lo, hi = np.percentile(stats, [2.5, 97.5])
    return float(np.mean(stats)), float(lo), float(hi)

# Concordance Correlation Coefficient for regression
def ccc(a, b):
    a, b = np.asarray(a), np.asarray(b)
    mu_a, mu_b = a.mean(), b.mean()
    var_a, var_b = a.var(), b.var()
    cov = np.cov(a, b, bias=True)[0,1]
    return 2*cov/(var_a + var_b + (mu_a - mu_b)**2 + 1e-12)

# Dummy loaders for a dry run
def load_dummy_classification(n_speakers=30, samples_per_spk=20, n_features=32):
    rng = np.random.default_rng(0)
    spk = np.repeat(np.arange(n_speakers), samples_per_spk)
    X = rng.normal(size=(len(spk), n_features))
    y_spk = (rng.normal(size=n_speakers) > 0).astype(int)
    y = y_spk[spk]
    return X, y, spk

def load_dummy_regression(n_speakers=30, samples_per_spk=20, n_features=32):
    rng = np.random.default_rng(1)
    spk = np.repeat(np.arange(n_speakers), samples_per_spk)
    X = rng.normal(size=(len(spk), n_features))
    beta = rng.normal(size=n_features)
    y = X @ beta + rng.normal(scale=0.5, size=len(spk))
    return X, y, spk

In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from sklearn.metrics import roc_auc_score, average_precision_score, f1_score

# Replace with your real features and labels
X, y, speaker_id = load_dummy_classification()

pipe = Pipeline([
    ("scaler", StandardScaler()),
    ("clf", SVC(C=1, gamma="scale", class_weight="balanced", probability=True))
])

cv = GroupKFold(n_splits=5)
probs = np.zeros(len(y))
for tr, te in cv.split(X, y, groups=speaker_id):
    pipe.fit(X[tr], y[tr])
    probs[te] = pipe.predict_proba(X[te])[:,1]

# Aggregate per speaker to match typical usage
probs_df = pd.DataFrame({"spk": speaker_id, "y": y, "p": probs})
agg = probs_df.groupby("spk").agg(y=("y","mean"), p=("p","mean"))
agg["y"] = (agg["y"] > 0.5).astype(int)

roc = roc_auc_score(agg["y"], agg["p"]) 
pr  = average_precision_score(agg["y"], agg["p"]) 
yhat = (agg["p"] >= 0.5).astype(int)
macro_f1 = f1_score(agg["y"], yhat, average="macro")
print({"ROC-AUC": roc, "PR-AUC": pr, "Macro-F1": macro_f1})

# Confidence intervals via bootstrap over speakers
mean_f1, lo_f1, hi_f1 = ci95_by_speaker(
    lambda yt, yp: f1_score(yt, (yp>=0.5).astype(int), average="macro"),
    agg["y"].values, agg["p"].values, agg.index.values)
print({"Macro-F1 mean": mean_f1, "95% CI": (lo_f1, hi_f1)})

In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVR
from sklearn.metrics import mean_absolute_error, root_mean_squared_error, r2_score

X, y, speaker_id = load_dummy_regression()

pipe = Pipeline([
    ("scaler", StandardScaler()),
    ("svr", SVR(C=10, epsilon=0.1, gamma="scale"))
])

cv = GroupKFold(n_splits=5)
preds = np.zeros_like(y, dtype=float)
for tr, te in cv.split(X, y, groups=speaker_id):
    pipe.fit(X[tr], y[tr])
    preds[te] = pipe.predict(X[te])

# Per‑speaker aggregation
_df = pd.DataFrame({"spk": speaker_id, "y": y, "p": preds})
agg = _df.groupby("spk").mean(numeric_only=True)
mae = mean_absolute_error(agg["y"], agg["p"]) 
rmse = root_mean_squared_error(agg["y"], agg["p"], squared=False)
cc = ccc(agg["y"].values, agg["p"].values)
r2 = r2_score(agg["y"], agg["p"]) 
print({"MAE": mae, "RMSE": rmse, "R2": r2, "CCC": cc})

# Confidence intervals
mean_mae, lo_mae, hi_mae = ci95_by_speaker(mean_absolute_error, agg["y"].values, agg["p"].values, agg.index.values)
print({"MAE mean": mean_mae, "95% CI": (lo_mae, hi_mae)})

In [ ]:
cv = GroupKFold(n_splits=5)
rows = []
for k, (_tr, te) in enumerate(cv.split(X, y, groups=speaker_id)):
    rows.append(pd.DataFrame({"idx": te, "fold": k, "speaker": speaker_id[te]}))
folds = pd.concat(rows, ignore_index=True)
folds.to_csv("folds_speaker_groupkfold.csv", index=False)
print("Saved folds_speaker_groupkfold.csv")

In [ ]:
# This utility walks a directory tree and computes per‑file MFCC mean/std.
# It assumes file names like <speaker>__<label>__anything.wav so it can
# recover speaker and label without a separate manifest.
import librosa, glob

def mfcc_stats_from_files(wav_dir, sr=16000, n_mfcc=20):
    rows, y, spk = [], [], []
    for path in glob.glob(os.path.join(wav_dir, "**", "*.wav"), recursive=True):
        base = os.path.basename(path)
        speaker, label, _rest = base.split("__", 2)
        x, _ = librosa.load(path, sr=sr, mono=True)
        x, _ = librosa.effects.trim(x, top_db=30)
        m = librosa.feature.mfcc(y=x, sr=sr, n_mfcc=n_mfcc)
        feat = np.concatenate([m.mean(axis=1), m.std(axis=1)])
        rows.append(feat); y.append(label); spk.append(speaker)
    X = np.vstack(rows).astype(float)
    y = np.array(y)
    spk = np.array(spk)
    return X, y, spk

---

## Appendix A — Core models, features, and metrics

This appendix deepens the theory while keeping the narrative style. It defines common speech features, staple learning algorithms, and evaluation metrics. Equations are now in LaTeX.

### A.1 From waveform to time–frequency: STFT, Mel spectrograms, and MFCCs

Start with a discrete‑time signal $x[n]$ sampled at $f_s$ Hz. Speech is quasi‑stationary over short windows, so we analyze it in frames. Choose a window $w[m]$ of length $N$ (e.g., Hann) and a hop $H$. The **short‑time Fourier transform (STFT)** is

$$
X(k,t)=\sum_{m=0}^{N-1} x[tH+m]\, w[m] \, e^{-j\,2\pi km/N}, \quad k=0,\ldots,N-1.
$$

The **power spectrogram** is $|X(k,t)|^2$, often displayed in dB.

Map frequency to the mel scale with

$$
\operatorname{mel}(f)=2595\,\log_{10}\!\left(1+\frac{f}{700}\right), \qquad f\;[\mathrm{Hz}].
$$

With $M$ triangular mel filters $\{h_m(k)\}_{m=1}^M$, the **mel band energies** are

$$
E_m(t)=\sum_{k=0}^{N-1} h_m(k)\, |X(k,t)|^2, \qquad m=1,\ldots,M,
$$

and the **log‑mel spectrogram** is $\log(E_m(t)+\varepsilon)$ with a small $\varepsilon>0$.

The **MFCCs** summarize spectral shape across mel bands. Stack $\mathbf e(t)=[\log E_1(t),\ldots,\log E_M(t)]^\top$ and apply a type‑II DCT:

$$
\mathrm{MFCC}_n(t)=\sum_{m=1}^{M} c_{n,m}\, \log E_m(t),\qquad n=0,\ldots,C-1,
$$

where for $n>0$,

$$
 c_{n,m}=\sqrt{\frac{2}{M}}\cos\!\left(\frac{\pi n}{M}\Big(m-\tfrac{1}{2}\Big)\right),\qquad c_{0,m}=\sqrt{\frac{1}{M}}.
$$

Time derivatives (**deltas**) are commonly computed by a local linear fit; a standard formula is

$$
\Delta c_t=\frac{\sum_{p=1}^P p\,\big(c_{t+p}-c_{t-p}\big)}{2\sum_{p=1}^P p^2}.
$$

For **utterance‑level features**, pool each coefficient over time by mean and standard deviation to obtain a compact vector suited to SVMs and tree ensembles.

Two frequently used STFT descriptors are the **spectral centroid** and **bandwidth** at time $t$:

$$
\operatorname{centroid}(t)=\frac{\sum_k f_k\,|X(k,t)|^2}{\sum_k |X(k,t)|^2},\qquad
\operatorname{bandwidth}(t)=\sqrt{\frac{\sum_k (f_k-\operatorname{centroid}(t))^2\,|X(k,t)|^2}{\sum_k |X(k,t)|^2}}.
$$

The **roll‑off** at percentile $\rho$ is the smallest $f_\rho$ whose cumulative energy reaches $\rho\%$ of the frame energy.

### A.2 Linear and kernel models in compact form

A **linear regressor** predicts

$$
\hat y=\mathbf w^\top\mathbf x + b.
$$

**Ridge** and **Lasso** minimize squared error with $\ell_2$ or $\ell_1$ regularization:

$$
\min_{\mathbf w,b}\; \sum_{i=1}^n \big(y_i-\mathbf w^\top\mathbf x_i-b\big)^2 + \lambda\,\|\mathbf w\|_2^2,\qquad
\min_{\mathbf w,b}\; \sum_{i=1}^n \big(y_i-\mathbf w^\top\mathbf x_i-b\big)^2 + \lambda\,\|\mathbf w\|_1.
$$

**Logistic regression** models

$$
P(y=1\mid \mathbf x)=\sigma(\mathbf w^\top\mathbf x + b)=\frac{1}{1+e^{-(\mathbf w^\top\mathbf x+b)}},
$$

with $\mathbf w,b$ chosen by minimizing the regularized negative log‑likelihood.

For **SVM classification**, the soft‑margin primal is

$$
\min_{\mathbf w,b,\boldsymbol\xi}\; \tfrac{1}{2}\,\|\mathbf w\|_2^2 + C\sum_{i=1}^n \xi_i\quad \text{s.t.}\quad y_i\,(\mathbf w^\top\mathbf x_i + b)\ge 1-\xi_i,\; \xi_i\ge 0.
$$

Kernelization replaces $\mathbf x^\top\mathbf x'$ with $K(\mathbf x,\mathbf x')$. The **RBF kernel** is

$$
K(\mathbf x,\mathbf x')=\exp\!\big(-\gamma\,\|\mathbf x-\mathbf x'\|_2^2\big).
$$

For **SVR**, an $\varepsilon$‑insensitive loss yields

$$
\min_{\mathbf w,b,\boldsymbol\xi,\boldsymbol\xi^*}\; \tfrac{1}{2}\,\\|\mathbf w\|_2^2 + C\sum_{i=1}^n (\xi_i+\xi_i^*)\quad \text{s.t.}\quad
\begin{cases}
 y_i-(\mathbf w^\top\mathbf x_i+b)\le \varepsilon+\xi_i,\\
 (\mathbf w^\top\mathbf x_i+b)-y_i\le \varepsilon+\xi_i^*,\\
 \xi_i,\xi_i^*\ge 0.
\end{cases}
$$

**Random Forests** average $T$ trees $\{f_t\}_{t=1}^T$ grown on bootstraps with feature subsetting:

$$
\hat y=\frac{1}{T}\sum_{t=1}^T f_t(\mathbf x)\;\; \text{(regression)},\qquad \hat y=\operatorname{mode}\{f_t(\mathbf x)\}\;\; \text{(classification)}.
$$

Randomization lowers variance; depth and leaf size control bias.

**Gradient‑Boosted Trees** form an additive model

$$
F_M(\mathbf x)=\sum_{m=1}^M \nu\, f_m(\mathbf x),
$$

where $f_m$ fits the negative gradient (pseudo‑residuals) of a chosen loss $\mathcal L$. Learning rate $\nu$, tree depth, and $M$ govern the bias–variance trade‑off; early stopping limits overfitting.

### A.3 Probability calibration

Raw scores need not be probabilities. **Platt scaling** fits

$$
P(y=1\mid s)=\frac{1}{1+e^{A s + B}}
$$

for validation scores $s$. **Isotonic regression** learns a nondecreasing mapping $g(s)$ from scores to probabilities. Calibration is summarized by the Brier score and visualized with reliability diagrams.

#### Venn–Abers calibration (probability intervals, not just point probabilities)

A useful alternative to Platt scaling and isotonic regression is **Venn–Abers (VA) calibration**, which combines isotonic regression with the **Venn prediction** framework to produce **well-calibrated probability estimates together with uncertainty**. Instead of outputting a single calibrated probability $\hat p$, VA treats the unknown label of a test point as a *hypothesis* $y\in\{0,1\}$. For each hypothesis, it temporarily assigns that label to the test point, fits an isotonic calibrator on the augmented calibration set (typically on the model scores $s$), and obtains a calibrated probability. This yields **two probabilities** $(p_0, p_1)$: one calibrated assuming the test label is 0, and one assuming it is 1. These can be interpreted as a **probability interval** $[ \min(p_0,p_1), \max(p_0,p_1) ]$ that reflects epistemic uncertainty due to finite calibration data. 

In practice, VA tends to be robust for small/medium calibration sets and avoids some pathologies of pure isotonic regression (overfitting and stepwise extremes), while still being nonparametric. When reporting results, state clearly whether you use the VA point estimate (often the midpoint) or keep the interval and propagate it to downstream decision rules.

### A.4 Metrics (formal definitions)

Let $\{(y_i,\hat y_i)\}_{i=1}^n$ be predictions; for probabilistic classifiers let $p_i=P(y=1\mid\mathbf x_i)$. The binary **confusion counts** are

$$
\mathrm{TP}=\sum \mathbb{1}[y_i=1,\ \hat y_i=1],\quad \mathrm{TN}=\sum \mathbb{1}[y_i=0,\ \hat y_i=0],\quad
\mathrm{FP}=\sum \mathbb{1}[y_i=0,\ \hat y_i=1],\quad \mathrm{FN}=\sum \mathbb{1}[y_i=1,\ \hat y_i=0].
$$

From these,

$$
\mathrm{Accuracy}=\frac{\mathrm{TP}+\mathrm{TN}}{n},\quad
\mathrm{Precision}=\frac{\mathrm{TP}}{\mathrm{TP}+\mathrm{FP}},\quad
\mathrm{Recall}=\frac{\mathrm{TP}}{\mathrm{TP}+\mathrm{FN}},\quad
\mathrm{Specificity}=\frac{\mathrm{TN}}{\mathrm{TN}+\mathrm{FP}}.
$$

The **F1** score is the harmonic mean

$$
\mathrm{F1}=\frac{2\,\mathrm{Precision}\cdot\mathrm{Recall}}{\mathrm{Precision}+\mathrm{Recall}}.
$$

For multi‑class tasks, **macro averaging** computes the unweighted mean of per‑class F1 values.

The **ROC curve** plots TPR against FPR $\big(\mathrm{FPR}=\tfrac{\mathrm{FP}}{\mathrm{FP}+\mathrm{TN}}\big)$ as the threshold varies; **ROC‑AUC** is its area. The **precision–recall (PR) curve** plots Precision against Recall; **PR‑AUC** summarizes it and is more informative under imbalance. The **Brier score** is

$$
\mathrm{Brier}=\frac{1}{n}\sum_{i=1}^n (p_i-y_i)^2.
$$

For regression with residuals $e_i=y_i-\hat y_i$,

$$
\mathrm{MAE}=\frac{1}{n}\sum |e_i|,\qquad \mathrm{MSE}=\frac{1}{n}\sum e_i^2,\qquad \mathrm{RMSE}=\sqrt{\mathrm{MSE}}.
$$

The coefficient of determination is

$$
R^2=1-\frac{\sum (y_i-\hat y_i)^2}{\sum (y_i-\bar y)^2}.
$$

When comparing to noisy human ratings or reference measures, the **Concordance Correlation Coefficient (CCC)** is

$$
\mathrm{CCC}=\frac{2\,\sigma_{y\hat y}}{\sigma_y^2+\sigma_{\hat y}^2+(\mu_y-\mu_{\hat y})^2},
$$

with sample means $\mu$, variances $\sigma^2$, and covariance $\sigma_{y\hat y}$. A **Bland–Altman** analysis plots differences $d_i=y_i-\hat y_i$ against means $m_i=(y_i+\hat y_i)/2$, reporting bias $\bar d$ and limits $\bar d\pm1.96\,s_d$.

In all cases where multiple utterances belong to a speaker, aggregate to speaker level before computing metrics to avoid overweighting talkative speakers and to align with the unit of decision.

### A.5 Conformal prediction with clean guarantees

Split training speakers into **proper‑train** and **calibration** sets. Fit a model on proper‑train. For regression, compute calibration residuals $r_i=|y_i-\hat f(\mathbf x_i)|$ and the $(1-\alpha)$ quantile

$$
q_{1-\alpha}=\operatorname{Quantile}_{1-\alpha}\{r_i\}_{i\in\mathrm{cal}}.
$$

For a new input $\mathbf x_\ast$, return the interval

$$
\big[\, \hat f(\mathbf x_\ast)-q_{1-\alpha},\;\hat f(\mathbf x_\ast)+q_{1-\alpha} \,\big],
$$

which attains finite‑sample coverage $\Pr\{y_\ast\in\cdot\}\ge 1-\alpha$ under exchangeability. For classification, choose a nonconformity score, e.g. $A(\mathbf x,y)=1-\hat p(y\mid\mathbf x)$, compute class‑wise (Mondrian) quantiles $q_{1-\alpha}^{(y)}$, and output the **prediction set**

$$
\mathcal C(\mathbf x_\ast)=\{\, y:\; A(\mathbf x_\ast,y)\le q_{1-\alpha}^{(y)} \,\}.
$$

Sets may legitimately contain more than one label when uncertainty is high.

### A.6 Why grouped (speaker‑independent) splits matter

Assume features factor approximately as $\phi(\mathbf x_{i,j})\approx g(\mathbf s_i)+h(\mathbf c_{i,j})$ with speaker trait $\mathbf s_i$ and content $\mathbf c_{i,j}$. If the same $\mathbf s_i$ appears in train and test, a flexible model can memorize $g$ and yield over‑optimistic estimates. Grouped splits remove this shortcut, forcing the model to learn $h$ that generalizes.

### A.7 Normalization, pipelines, and leakage

Standardization $z=(x-\mu)/\sigma$ must be estimated **only** on training data within each fold and then applied to validation/test. Scikit‑learn `Pipeline`s enforce this and prevent information leakage from validation/test back into training.

### A.8 Bias–variance and honest error bars

Forests and boosting reduce variance via averaging and stagewise fitting; SVM hyperparameters $C$ and $\gamma$ trade margin against fit. Cross‑validation estimates out‑of‑sample error but can be noisy with few speakers. Report **bootstrap confidence intervals over speakers**, and consider conformal prediction for per‑instance uncertainty. Together they keep results interpretable and honest.

### A.9 Jitter and Shimmer

**What they measure.**  
Classic **cycle-to-cycle perturbation** measures of sustained voiced phonation.

- **Jitter**: micro-fluctuations in the **fundamental period** (timing irregularity).  
- **Shimmer**: micro-fluctuations in **cycle amplitude** (amplitude irregularity).

They are most meaningful on **steady, sustained vowels** (e.g., /a/) at comfortable pitch/loudness for $\sim$ 1–3 s. On connected speech, intended prosody confounds them.

### A.9.1 Preliminaries and Notation

From a voiced segment, extract cycle boundaries (pitch marks) to form:

- Periods $\{T_i\}_{i=1}^N$ (in seconds).  
- Peak-to-peak (or RMS) amplitudes $\{A_i\}_{i=1}^N$ (linear units).

Let $$\bar T = \tfrac{1}{N}\sum_{i=1}^N T_i$$ and $$\bar A = \tfrac{1}{N}\sum_{i=1}^N A_i$$

> **Cycle extraction.** Start from a robust $F_0$ tracker; refine epochs near predicted instants by local extremum/zero-crossing search. If available, **EGG** (electroglottograph) improves reliability. Reject unvoiced frames and obvious outliers.


### A.9.2 Jitter (Period Perturbation)

1) **Local Jitter (relative, \%)**
$$\mathrm{Jitter}_{\mathrm{local}}(\%) \;=\; 100 \times 
\frac{\tfrac{1}{N-1}\sum_{i=1}^{N-1} \lvert T_{i+1}-T_i\rvert}{\bar T}$$

2) **Absolute Jitter (s or ms)**
$$\mathrm{Jitter}_{\mathrm{abs}} \;=\; \frac{1}{N-1}\sum_{i=1}^{N-1}\lvert T_{i+1}-T_i\rvert$$

3) **RAP (Relative Average Perturbation, \%)** — 3-cycle smoothing:
$$\mathrm{RAP}(\%) \;=\; 100 \times \frac{\tfrac{1}{N-2}\sum_{i=2}^{N-1} \Big\lvert\, T_i - \tfrac{T_{i-1}+T_i+T_{i+1}}{3}\,\Big\rvert}{\bar T}$$

4) **PPQ5 (Five-Point Period Perturbation Quotient, \%)** — 5-cycle smoothing:
$$\mathrm{PPQ5}(\%) \;=\; 100 \times \frac{\tfrac{1}{N-4}\sum_{i=3}^{N-2} \Big\lvert\, T_i - \tfrac{1}{5}\sum_{k=i-2}^{i+2} T_k\,\Big\rvert}{\bar T}$$

5) **DDP (Degree of Dysphonia — Period)**  
$$\mathrm{DDP} \;=\; 3 \times \mathrm{RAP}$$

**Notes.** Relative measures (\%) are unit-invariant; RAP/PPQ reduce sensitivity to slow $F_0$ drift by comparing to a local mean.


### A.9.3 Shimmer (Amplitude Perturbation)

Define $A_i$ consistently (peak-to-peak or RMS per cycle) using identical windows/landmarks across cycles.

1) **Local Shimmer (relative, \%)**
$$\mathrm{Shimmer}_{\mathrm{local}}(\%) = 100 \times \frac{\tfrac{1}{N-1}\sum_{i=1}^{N-1} \lvert A_{i+1}-A_i\rvert}{\bar A}$$

2) **Shimmer (dB)**  (often reported as mean absolute dB difference)
$$\mathrm{Shimmer}_{\mathrm{dB}} \;=\; \frac{1}{N-1}\sum_{i=1}^{N-1} \Bigl|\,20\log_{10}\!\frac{A_{i+1}}{A_i}\Bigr|$$

3) **APQ k (Amplitude Perturbation Quotient, \%)** — local $k$-cycle smoothing:
$$\mathrm{APQ}k(\%) \;=\; 100 \times 
\frac{\tfrac{1}{N-k+1}\sum_{i=\lceil k/2\rceil}^{N-\lfloor k/2\rfloor} 
\Bigl|\,A_i - \tfrac{1}{k}\!\!\sum_{r=i-\lfloor k/2\rfloor}^{i+\lceil k/2\rceil-1} \!\!A_r\,\Bigr|}
{\bar A}$$
Common: $\mathrm{APQ3}, \mathrm{APQ5}, \mathrm{APQ11}$.

4) **DDA (Degree of Dysphonia — Amplitude)**  
$$\mathrm{DDA} \;=\; 3 \times \mathrm{APQ3}.$$

**Notes.** Prefer relative (\%) forms to reduce dependence on overall gain. Keep the amplitude definition fixed across cycles/subjects.

### A.9.4 Practical Computation Details

- **Signal quality.** Use $\ge 16\,\mathrm{kHz}$, avoid clipping/AGC/compression. EGG markedly improves cycle marking.  

- **Cycle detection.** Track $F_0$, refine epochs within $\pm 20\%$ of predicted instants; reject outliers via robust rules (e.g., MAD).  

- **Windowing.** Analyze the central steady portion (e.g., 30–50 ms sliding windows) and summarize with medians/IQR to mitigate onset/offset transients.  

- **Amplitude choice.** Peak-to-peak or RMS; **do not** normalize amplitude per cycle (that erases shimmer).  

- **Neighborhood sizes.** RAP/APQ3 (3-cycle), PPQ5/APQ5 (5-cycle) are common; APQ11 provides stronger smoothing for longer vowels.  

- **Reporting.** Always state: extractor (PRAAT-style, YIN/CREPE+epoch refinement, EGG), variants (e.g., Jitter local \%, RAP, PPQ5; Shimmer local \%, APQ5), window length, voiced criteria, and outlier handling.



### A.9.5 Interpretation, Confounds, Recommendations

- **Interpretation.** Higher jitter $\Rightarrow$ timing irregularity of vocal fold vibration; higher shimmer $\Rightarrow$ amplitude irregularity.

- **Confounds.** $F_0$ errors inflate jitter; prosodic drift raises local jitter (RAP/PPQ help but don’t fully remove); distance/AGC/compression bias shimmer; creaky/aperiodic voice can destabilize all measures.  

- **Best practice.** Use sustained vowels with consistent mouth–mic distance; report at least Jitter $_{\mathrm{local}}$, **RAP**, **PPQ5**, Shimmer $_{\mathrm{local}}$, **APQ5**; add **HNR** or **GNE** to complement shimmer.


### A.9.6 Minimal Algorithmic Sketch (Acoustic Waveform)

1. Band-limit (e.g., $50–4000$ Hz), de-DC; optional mild pre-emphasis.  
2. Track $F_0$ on voiced frames; interpolate short gaps.  
3. Place/refine pitch marks near expected epochs.  
4. Compute $T_i$ from successive marks; reject implausible values (e.g., outside $[0.5,2]\times$ local median).  
5. Compute $A_i$ (peak-to-peak or RMS) from consistent per-cycle windows.  
6. Compute indices (local, RAP/PPQ, APQ$k$); summarize (median, IQR) over the central steady segment.

### A.9.7 Quick Reference

- Jitter $_{\mathrm{local}}$ (\%), **RAP** (\%), **PPQ5** (\%): period irregularity.  
- Jitter $_{\mathrm{abs}}$ (ms): absolute period difference.  
- Shimmer $_{\mathrm{local}}$ (\%), **APQ k** (\%), **Shimmer(dB)**: amplitude irregularity.  
- **DDP / DDA**: historical MDVP composites, $3\times\mathrm{RAP}$ and $3\times\mathrm{APQ3}$.

*Treat these as protocol-sensitive acoustic biomarkers; always ship the extraction recipe with the numbers.*

## A.10 Glottal Source Signals, Inverse Filtering, and Glottal Features (LP / IAIF)

This section is about **estimating the glottal excitation** (or a proxy of it) from recorded speech, and then extracting **source-related features** that are often more “physiological” than purely acoustic features. It is especially useful when you care about **phonation quality** (breathy/pressed/tense), vocal efficiency, or laryngeal dynamics.

---

### A.10.1 Source–filter model: what we are trying to invert

A standard working model for voiced speech is the **linear source–filter** model:

$$ s[n] \approx (g[n] * v[n]) * r[n] $$

where

- $s[n]$ is the recorded speech signal (after mic + room effects; assume we’ve minimized these),
- $g[n]$ is the **glottal flow** (volume velocity) waveform at the glottis,
- $v[n]$ is the **vocal tract impulse response** (formant filter),
- $r[n]$ is the **radiation characteristic** at the lips (often approximated by a differentiator).

Many pipelines work with the **glottal flow derivative** rather than the flow itself. The radiation at the lips is often approximated as

$$
r[n] \approx \delta[n] - \alpha\,\delta[n-1] \quad \Rightarrow \quad R(z) \approx 1-\alpha z^{-1},
$$

so lip radiation roughly differentiates the flow. A common simplification is:

$$
s[n] \approx \dot{g}[n] * v[n],
$$

i.e., speech is approximately the convolution of a **source derivative** with the vocal tract filter. The purpose of **inverse filtering** is to remove $v[n]$ (and sometimes undo radiation) to recover an estimate of $g[n]$ or $\dot{g}[n]$.

---

### A.10.2 What is a “glottal signal” in practice?

Depending on the method, you may estimate one of the following (names vary across toolkits/papers):

- **Estimated glottal flow**: $\hat{g}[n]$  
- **Estimated glottal flow derivative**: $\widehat{\dot{g}}[n]$  
- A related “source” waveform from which cycle measures are extracted (e.g., the waveform used to locate the **GCI**—glottal closure instants).

Important practical point: **absolute scaling is usually not identifiable** from microphone speech alone. Many glottal features are therefore defined as **ratios** or **time-normalized** descriptors.

---

### A.10.3 Inverse filtering by Linear Prediction (LP): the core idea

**Linear prediction** models the short-time speech signal as an all-pole autoregressive (AR) process:

$$
\hat{s}[n] = \sum_{k=1}^{p} a_k\, s[n-k], 
\quad
e[n] = s[n]-\hat{s}[n].
$$

This corresponds to an all-pole transfer function

$$
A(z) = 1 - \sum_{k=1}^{p} a_k z^{-k},
\quad
H_{\text{LP}}(z)=\frac{1}{A(z)}.
$$

For a windowed voiced frame, if LP predominantly captures the **vocal tract resonances**, then the **prediction error** $e[n]$ becomes a proxy for the **excitation**. In classic LPC vocoding, $e[n]$ is the residual. In glottal inverse filtering, we are more careful because:

- The glottal source is **not white** and is itself shaped (spectral tilt).
- Strong source–tract interaction and imperfect modeling can leak source information into the tract estimate and vice versa.

Still, LP is the foundation: “estimate tract $\Rightarrow$ inverse-filter speech $\Rightarrow$ get a source-like residual”.

**Key tunables**
- $p$: LP order (often $p \approx 2 + \frac{f_s}{1000}$ for speech; e.g., $p\sim 16$ at 16 kHz is common, but glottal work often uses careful heuristics).
- frame length (typically 20–40 ms) and windowing.
- pre-emphasis and/or DC removal.

---

### A.10.4 IAIF: Iterative Adaptive Inverse Filtering (LP-based but smarter)

**IAIF** is a widely used method for estimating glottal flow from microphone speech, especially in Finnish/Alku-group style pipelines. Its philosophy:

1) **Remove lip radiation** (approximate integration, because radiation is like differentiation).  
2) Estimate a **coarse glottal contribution**, remove it.  
3) Estimate a better **vocal tract filter** from the “deglottalized” signal.  
4) Re-estimate glottal flow using the improved tract estimate.

It is “iterative” because it alternates between source and tract estimation.

A simplified conceptual outline (not strict implementation details, which vary):

1. **Preprocessing**: remove DC, optionally high-pass; optionally compensate radiation by an integrator $1/(1-\beta z^{-1})$.
2. **First-pass glottal model**: low-order LP (small $p_g$, e.g., 2–4) to capture gross spectral tilt of the source.
3. **First-pass tract model**: higher-order LP (e.g., $p_v \sim 10$–20 depending on $f_s$) on the partly deglottalized signal.
4. **Second-pass glottal model**: re-estimate source on the signal inverse-filtered by the improved tract estimate; possibly integrate to obtain flow.

Why two LP orders? Because:
- The vocal tract is relatively high-order (multiple formants).
- The glottal source spectral shape (tilt) is low-order and smooth.

A simple way to express the decomposition:

$$
S(z) \approx G(z)\,V(z)\,R(z).
$$

IAIF attempts to estimate $V(z)$ and $G(z)$ with two all-pole models (different orders), alternating so each estimate is less contaminated by the other.

**Practical note.** IAIF works best on:
- Sustained vowels or stable voiced segments,
- Good SNR recordings,
- Correct pitch tracking / voiced segmentation.

---

### A.10.5 From estimated glottal flow to glottal events (GCI, GOI)

Many glottal features depend on identifying:
- **GCI**: glottal closure instant (often near the maximum negative peak in $\widehat{\dot{g}}[n]$),
- **GOI**: glottal opening instant (harder to locate reliably).

Let one glottal cycle be delimited by consecutive closures:
$$
T_0 = t_{c}^{(i+1)} - t_{c}^{(i)}.
$$

Within that cycle, define:
- Open phase duration $T_o$,
- Closed phase duration $T_c$,
- Return phase duration $T_r$ (closing return), etc., depending on the chosen model.

Even if you do not explicitly locate GOI, many robust features are defined using landmarks on $\hat{g}[n]$ or $\widehat{\dot{g}}[n]$.

---

### A.10.6 Common glottal features with definitions and equations

Below are features often reported in phonation analysis. Definitions vary slightly across publications and toolkits; I’ll state the **most common analytical definitions** and note the typical variants.

#### (1) $F_0$ and cycle timing from the source estimate

Fundamental period:
$$
T_0 = t_{c}^{(i+1)} - t_{c}^{(i)}, 
\quad
F_0 = \frac{1}{T_0}.
$$

These are usually computed from GCI sequences detected from $\widehat{\dot{g}}[n]$ or from the speech itself.

---

#### (2) Open Quotient (OQ / QOQ)

**Concept.** Fraction of the cycle during which the glottis is “open”.

If you can estimate glottal opening and closure instants (GOI and GCI):
$$
\mathrm{OQ} = \frac{T_o}{T_0} = \frac{t_{\text{close}} - t_{\text{open}}}{T_0}.
$$

**QOQ (Quasi-Open Quotient)** is often used when GOI is unreliable. A common operational definition uses the glottal flow $\hat{g}(t)$ and defines the “open” interval as where $\hat{g}(t)$ exceeds some fraction $\theta$ of its cycle maximum:

Let $g_{\max}$ be the maximum glottal flow in the cycle. Define open set:
$$
\mathcal{O} = \{t \in \text{cycle}: \hat{g}(t) \ge \theta\,g_{\max}\},
$$
then
$$
\mathrm{QOQ}(\theta)=\frac{\text{duration}(\mathcal{O})}{T_0}.
$$

Typical choices: $\theta = 0.5$ (50%) or other fixed fractions, but toolkits differ. Always report $\theta$.

**Interpretation.**
- Larger OQ/QOQ → more open/breathy phonation (often; but be cautious with interaction effects).
- Smaller OQ → more pressed/abrupt closure.

---

#### (3) Speed Quotient (SQ)

**Concept.** Ratio of opening time to closing time within the open phase.

Let $T_{\text{open}}$ be from opening to peak flow, and $T_{\text{close}}$ be from peak flow to closure (within the open portion). Then

$$
\mathrm{SQ} = \frac{T_{\text{open}}}{T_{\text{close}}}.
$$

**Interpretation.**
- $\mathrm{SQ} > 1$: slower opening, faster closing (more abrupt closure tendency).
- $\mathrm{SQ} < 1$: fast opening, slow closing.

Variants: Some define SQ using different landmark points (e.g., 50% amplitude crossings). State your landmarking rule.

---

#### (4) Closing Quotient (CQ)

Another related time ratio:

$$
\mathrm{CQ} = \frac{T_{\text{close}}}{T_0},
$$

where $T_{\text{close}}$ is the closing (return) phase duration (peak to closure) or a defined closing subinterval.

---

#### (5) NAQ (Normalized Amplitude Quotient)

NAQ is widely used because it is relatively robust and correlates with phonation type.

Let:
- $A_{\text{pp}}$ be the **peak-to-peak amplitude** of the glottal flow estimate within a cycle:
  $$
  A_{\text{pp}} = g_{\max} - g_{\min}.
  $$
  (Often $g_{\min}$ is near zero; depends on baseline.)
- $\mathrm{MFDR}$ be the **maximum flow declination rate**, i.e., the magnitude of the most negative slope of flow during closing:
  $$
  \mathrm{MFDR} = \max_{t\in\text{cycle}} \left( -\frac{d\hat{g}(t)}{dt} \right).
  $$
  If you work directly with flow derivative $\widehat{\dot{g}}(t)$, then $\mathrm{MFDR}$ is essentially the magnitude of its negative peak (depending on sign conventions).

Then:

$$
\mathrm{NAQ} = \frac{A_{\text{pp}}}{T_0 \cdot \mathrm{MFDR}}.
$$

**Interpretation.**
- Higher NAQ is often associated with **breathier** phonation (less abrupt closing, smaller MFDR relative to amplitude).
- Lower NAQ often indicates **tenser/pressed** phonation (more abrupt closing, large MFDR).

Important: NAQ depends on a decent estimate of both $g(t)$ and its slope near closure, so quality of inverse filtering matters.

---

#### (6) H1–H2 and harmonic measures (source spectral tilt proxy)

Although not strictly “glottal-flow” features, **source tilt** is often measured via harmonic amplitudes.

Let $H1$ be the amplitude of the first harmonic at $F_0$ and $H2$ the amplitude at $2F_0$, measured on a spectrum of a windowed voiced segment (or on $\hat{g}(t)$/$\widehat{\dot{g}}(t)$). Then:

$$
\mathrm{H1{-}H2} = 20\log_{10}\frac{|S(F_0)|}{|S(2F_0)|}.
$$

Higher H1–H2 usually corresponds to **more spectral tilt** (often breathier voice), but formant interactions can contaminate this in raw speech; measuring on the inverse-filtered source is often cleaner.

---

#### (7) PSP (Peak Slope Parameter) — slope near closure

PSP is intended to capture how “sharp” the closing event is, related to high-frequency excitation. Definitions vary. A common family of definitions uses the derivative waveform around closure:

- Let $t_c$ be the GCI location within the cycle.
- Consider a short neighborhood $[t_c-\Delta, t_c+\Delta]$.

One operational definition is:

$$
\mathrm{PSP} = \frac{\max_{t\in\mathcal{W}} \left| \widehat{\dot{g}}(t) \right|}{A_{\text{pp}}},
$$

where $\mathcal{W}$ is a window around closure and $A_{\text{pp}}$ is flow peak-to-peak. Some formulations normalize by $T_0$ or by RMS energy. The intent is: **larger closure-related derivative peaks** imply sharper closures.

Because PSP is not as standardized as NAQ/OQ, you should explicitly state:
- whether computed from $\hat{g}$ or $\widehat{\dot{g}}$,
- window size $\Delta$,
- normalization.

---

#### (8) AC Flow (Amplitude of the AC component of flow)

AC flow is typically the amplitude of the **periodic component** of $\hat{g}(t)$. A common cycle definition:

$$
\mathrm{AC} = g_{\max}-g_{\min} = A_{\text{pp}}.
$$

Sometimes AC is defined as RMS of the demeaned flow within a cycle:
$$
\mathrm{AC}_{\mathrm{RMS}} = \sqrt{\frac{1}{T_0}\int_{\text{cycle}} (g(t)-\bar g)^2\,dt}.
$$

---

#### (9) HRF / spectral tilt parameters on the glottal source

If you compute a spectrum of the estimated source, you can summarize tilt by a regression slope:

Let $P(f)=20\log_{10}|G(f)|$ on a frequency range $[f_1,f_2]$. Fit
$$
P(f) \approx af + b,
$$
then $a$ is the **tilt slope** (dB/Hz). Sometimes a log-frequency axis is used. These features are closely related to phonation type.

---

### A.10.7 What can go wrong (and how to interpret results skeptically)

1) **Pitch marking errors** (wrong GCI) will break cycle features (OQ, NAQ, etc.).  
2) **Inverse filtering leakage**: if tract estimation captures part of the source, $\hat{g}$ becomes distorted; NAQ and OQ may shift systematically.  
3) **Formant proximity to harmonics**: in raw speech, harmonic amplitudes (H1, H2) can be shaped by formants; inverse filtering helps but is imperfect.  
4) **Source–tract interaction**: the separation is not perfectly linear, especially for high effort/pressed voice.  
5) **Recording chain** (mic, AGC, compression) can alter slopes and amplitudes; prefer ratio-based features and controlled recording.

**Good practice.**
- Prefer stable sustained vowels for glottal features.
- Inspect a few inverse-filtered waveforms per speaker: does $\hat{g}(t)$ look plausible (smooth rise, clear closure)?  
- Report the inverse filtering method (LP order, IAIF settings, radiation compensation) alongside the features.

---

### A.10.8 Minimal “feature recipe” for a robust glottal feature set

If you want a compact, defensible set that often behaves well:

- $F_0$ and $T_0$ statistics from GCI sequence,
- $\mathrm{NAQ}$,
- $\mathrm{QOQ}(\theta)$ with a stated $\theta$ (e.g., 0.5),
- $\mathrm{SQ}$ (with stated landmarks),
- optional: spectral tilt proxy on $\hat{g}$ or $\widehat{\dot{g}}$ (e.g., slope or H1–H2 on the estimated source).

These capture **timing**, **open/closed balance**, and **closure abruptness**—the three pillars of many phonation-type analyses.

---

### A.10.9 A note on definitions across toolkits

Features like NAQ and QOQ are relatively standardized, but PSP and even “OQ” can differ depending on whether GOI is explicitly estimated or inferred via thresholds. Whenever you publish or share results, include:

- exact mathematical definition (as above),
- cycle landmarking method (GCI/GOI logic),
- and all hyperparameters (threshold $\theta$, window sizes, LP orders).

In [ ]:
"""
Simple glottal inverse filtering pipeline (LP / IAIF-style) over a folder of .WAV files.

What you get:
- A crude estimate of glottal flow derivative (LF-like source proxy) via inverse filtering
- Optionally an integrated “glottal flow” estimate
- A few sanity plots for one file

This is *not* a drop-in replacement for a mature IAIF implementation (e.g., Aalto/Alku toolchains),
but it is a transparent baseline that illustrates the steps and produces reasonable outputs on
clean sustained vowels / stable voiced segments.

Dependencies:
    pip install numpy scipy librosa soundfile matplotlib

Usage:
    python this_script.py  (or paste into a notebook cell and run)
"""

from __future__ import annotations
import os, glob
import numpy as np
import soundfile as sf
import librosa
from scipy.signal import lfilter
import matplotlib.pyplot as plt


# -----------------------------
# Utilities: LPC and filtering
# -----------------------------

def lpc_allpole_coeffs(x: np.ndarray, order: int) -> np.ndarray:
    """
    Returns LPC polynomial A(z) = 1 + a1 z^-1 + ... + ap z^-p
    librosa.lpc returns that convention (leading 1).
    """
    # librosa expects float64 and finite values
    x = np.asarray(x, dtype=np.float64)
    x = np.nan_to_num(x)
    # guard: too short
    if len(x) < order + 2:
        raise ValueError("Signal too short for requested LPC order.")
    a = librosa.lpc(x, order=order)
    return a


def inverse_filter(x: np.ndarray, A: np.ndarray) -> np.ndarray:
    """
    Applies inverse filtering using LPC polynomial A(z).
    If X(z) ~ 1/A(z) * E(z), then E(z) ~ A(z) * X(z).
    """
    return lfilter(A, [1.0], x)


def leaky_integrator(x: np.ndarray, beta: float = 0.99) -> np.ndarray:
    """
    Approximate integration: y[n] = beta*y[n-1] + x[n]
    Transfer: 1 / (1 - beta z^-1)
    """
    return lfilter([1.0], [1.0, -beta], x)


def leaky_differentiator(x: np.ndarray, alpha: float = 0.99) -> np.ndarray:
    """
    Approximate differentiation: y[n] = x[n] - alpha*x[n-1]
    Transfer: 1 - alpha z^-1
    """
    return lfilter([1.0, -alpha], [1.0], x)


# -----------------------------------------
# IAIF-style inverse filtering (simplified)
# -----------------------------------------

def iaif_simple(
    x: np.ndarray,
    sr: int,
    frame_ms: float = 40.0,
    hop_ms: float = 10.0,
    pv: int | None = None,   # vocal tract LPC order
    pg1: int = 3,            # coarse glottal LPC order (pass 1)
    pg2: int = 4,            # refined glottal LPC order (pass 2)
    rad_beta: float = 0.99,  # leaky integrator for radiation compensation
    hp: float = 30.0,        # high-pass via spectral trim is crude; we just DC-remove
) -> dict:
    """
    Returns a dict with time-domain signals:
      - x: input (DC-removed)
      - x_rad: radiation-compensated (integrated)
      - e_glottal_1: coarse glottal-removed signal (after inverse filtering by Ag1)
      - e_vt: vocal-tract residual (after inverse filtering by Av)
      - g_der: estimated glottal flow derivative (after inverse filtering by Ag2)
      - g_flow: estimated glottal flow (integrated g_der)
    """

    x = np.asarray(x, dtype=np.float64)
    x = x - np.mean(x)

    # radiation compensation: speech ~ d/dt (g) * V  => integrate to “undo” radiation-ish differentiation
    x_rad = leaky_integrator(x, beta=rad_beta)

    # choose a reasonable vt LPC order if not provided
    # rule-of-thumb: ~2 formants per kHz + 2; at 16 kHz => ~18-20
    if pv is None:
        pv = int(2 + sr / 1000)  # e.g., 18 at 16 kHz

    # frame parameters
    frame_len = int(round(frame_ms * 1e-3 * sr))
    hop_len = int(round(hop_ms * 1e-3 * sr))
    frame_len = max(frame_len, pv + 5)
    hop_len = max(1, hop_len)

    # We do analysis on voiced-ish frames. For a minimal example, we process all frames and
    # overlap-add the residuals (this is simple, not perfect).
    win = np.hanning(frame_len).astype(np.float64)

    # output buffers
    e_vt = np.zeros_like(x_rad)
    g_der = np.zeros_like(x_rad)
    ola_w = np.zeros_like(x_rad) + 1e-12  # to normalize overlap-add

    for start in range(0, len(x_rad) - frame_len + 1, hop_len):
        seg = x_rad[start:start + frame_len] * win

        # --- Pass 1: coarse glottal model (low-order LPC) ---
        Ag1 = lpc_allpole_coeffs(seg, order=pg1)
        # remove coarse glottal shaping (inverse filter by Ag1)
        seg1 = inverse_filter(seg, Ag1)

        # --- Estimate vocal tract (higher-order LPC) on coarse-deglottalized signal ---
        Av = lpc_allpole_coeffs(seg1, order=pv)
        # VT residual (excitation proxy)
        seg_vt_res = inverse_filter(seg, Av)

        # --- Pass 2: refined glottal model on VT-removed signal ---
        # In IAIF, you estimate glottal model on the inverse-VT filtered signal (source side)
        # Here, seg_vt_res is already “source-ish”; fit low-order LPC to capture tilt and
        # take its inverse-filtered output as glottal derivative proxy.
        Ag2 = lpc_allpole_coeffs(seg_vt_res, order=pg2)
        seg_g_der = inverse_filter(seg_vt_res, Ag2)

        # Overlap-add
        e_vt[start:start + frame_len] += seg_vt_res * win
        g_der[start:start + frame_len] += seg_g_der * win
        ola_w[start:start + frame_len] += (win * win)

    # normalize OLA
    e_vt = e_vt / ola_w
    g_der = g_der / ola_w

    # flow is integral of derivative
    g_flow = leaky_integrator(g_der, beta=0.995)

    return {
        "x": x,
        "x_rad": x_rad,
        "e_vt": e_vt,
        "g_der": g_der,
        "g_flow": g_flow,
        "sr": sr,
        "params": {
            "frame_ms": frame_ms, "hop_ms": hop_ms, "pv": pv, "pg1": pg1, "pg2": pg2,
            "rad_beta": rad_beta
        }
    }


# -----------------------------
# Folder runner + I/O
# -----------------------------

def process_folder(
    wav_dir: str,
    out_dir: str,
    target_sr: int = 16000,
    save_outputs: bool = True,
    plot_one: bool = True
):
    os.makedirs(out_dir, exist_ok=True)
    wav_paths = sorted(glob.glob(os.path.join(wav_dir, "**", "*.wav"), recursive=True))
    if not wav_paths:
        raise FileNotFoundError(f"No .wav files found under: {wav_dir}")

    print(f"Found {len(wav_paths)} wav files.")

    example_done = False

    for p in wav_paths:
        x, sr = sf.read(p)
        if x.ndim > 1:
            x = np.mean(x, axis=1)
        if sr != target_sr:
            x = librosa.resample(x.astype(np.float64), orig_sr=sr, target_sr=target_sr)
            sr = target_sr

        # light trim (optional) – good for sustained vowels; remove if you want strict processing
        x, _ = librosa.effects.trim(x, top_db=30)

        out = iaif_simple(x, sr)

        base = os.path.splitext(os.path.basename(p))[0]
        if save_outputs:
            # Save glottal derivative proxy and flow estimate as WAV for inspection
            # Normalize to avoid clipping (these are not physically calibrated!)
            def _norm(y):
                m = np.max(np.abs(y)) + 1e-12
                return 0.95 * (y / m)

            sf.write(os.path.join(out_dir, f"{base}__g_der.wav"), _norm(out["g_der"]), sr)
            sf.write(os.path.join(out_dir, f"{base}__g_flow.wav"), _norm(out["g_flow"]), sr)
            sf.write(os.path.join(out_dir, f"{base}__vt_residual.wav"), _norm(out["e_vt"]), sr)

        if plot_one and not example_done:
            example_done = True
            t = np.arange(len(out["x"])) / sr

            plt.figure()
            plt.plot(t, out["x"])
            plt.title("Input speech (DC-removed)")
            plt.xlabel("Time [s]")
            plt.ylabel("Amplitude")

            plt.figure()
            plt.plot(t, out["x_rad"])
            plt.title("After radiation compensation (leaky integration)")
            plt.xlabel("Time [s]")

            plt.figure()
            plt.plot(t, out["e_vt"])
            plt.title("Vocal-tract inverse-filter residual (excitation proxy)")
            plt.xlabel("Time [s]")

            plt.figure()
            plt.plot(t, out["g_der"])
            plt.title("Estimated glottal flow derivative (IAIF-style, simplified)")
            plt.xlabel("Time [s]")

            plt.figure()
            plt.plot(t, out["g_flow"])
            plt.title("Estimated glottal flow (integrated derivative)")
            plt.xlabel("Time [s]")

            plt.show()

        print(f"Processed: {p}")

    print(f"Done. Outputs saved to: {out_dir}")


# -----------------------------
# Example call
# -----------------------------
if __name__ == "__main__":
    WAV_DIR = "/path/to/your/wavs"        # <-- change this
    OUT_DIR = "./glottal_outputs"        # <-- change this
    process_folder(WAV_DIR, OUT_DIR, target_sr=16000, save_outputs=True, plot_one=True)